# 8. JSON storage with web API

This is an example of using JSON with web API.

__Installing dependencies__

In [1]:
!python3 -m pip install -q dff[examples]

__Running example__

In [2]:
import pathlib

from dff.context_storages import context_storage_factory

from dff.pipeline import Pipeline
from dff.utils.testing.common import check_happy_path, is_interactive_mode
from dff.utils.testing.toy_script import TOY_SCRIPT, HAPPY_PATH

from flask import Flask, request

In [3]:
app = Flask(__name__)

pathlib.Path("dbs").mkdir(exist_ok=True)
db = context_storage_factory("json://dbs/file.json")


@app.route("/chat", methods=["GET", "POST"])
def respond():
    user_id = str(request.values.get("id"))
    user_message = str(request.values.get("message"))
    context = pipeline(user_message, user_id)
    return {"response": str(context.last_response)}

In [4]:
pipeline = Pipeline.from_script(
    TOY_SCRIPT,
    context_storage=db,
    start_label=("greeting_flow", "start_node"),
    fallback_label=("greeting_flow", "fallback_node"),
)

In [5]:
if __name__ == "__main__":
    check_happy_path(pipeline, HAPPY_PATH)
    if is_interactive_mode():
        app.run(
            host="0.0.0.0", port=5000, debug=True
        )  # This runs example in interactive mode (via flask, as a web server)

(user) >>> Hi
 (bot) <<< Hi, how are you?
(user) >>> i'm fine, how are you?
 (bot) <<< Good. What do you want to talk about?
(user) >>> Let's talk about music.
 (bot) <<< Sorry, I can not talk about music now.
(user) >>> Ok, goodbye.
 (bot) <<< bye
(user) >>> Hi
 (bot) <<< Hi, how are you?
